In [ ]:
from pathlib import Path
import random
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import Image, display

import torch
from ultralytics import YOLO

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
DATA_ROOT = Path("data")

train_txt = DATA_ROOT / "train.txt"
val_txt = DATA_ROOT / "validation.txt"
test_txt = DATA_ROOT / "test.txt"

def read_split(txt_path):
    return [
        Path(line.strip())
        for line in txt_path.read_text().splitlines()
        if line.strip()
    ]


def label_path(img_path):
    return img_path.with_suffix(".txt")


def print_split_info(split_name, images):
    existing_images = sum(p.exists() for p in images)
    existing_labels = sum(label_path(p).exists() for p in images)
    print(f"{split_name:<5}: {len(images)} {existing_images} {existing_labels}")


train_images = read_split(train_txt)
val_images = read_split(val_txt)
test_images = read_split(test_txt)

for split_name, images in [("train", train_images), ("val", val_images), ("test", test_images)]:
    print_split_info(split_name, images)

In [ ]:
DATA_YAML = Path("hrplanes.yaml")

print(DATA_YAML.read_text())

In [ ]:
MODEL_NAME = "yolov8s.pt"

IMG_SIZE = 640
BATCH_SIZE = 32
NUM_EPOCHS = 50
NUM_WORKERS = 8

LR = 1e-3
TARGET_MAP50 = 0.70

DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


In [ ]:
sample_images = train_images[:6]

plt.figure(figsize=(12, 6))

for i, img_path in enumerate(sample_images):
    img = PILImage.open(img_path).convert("RGB")
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=NUM_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    optimizer="AdamW",
    lr0=LR,
    workers=NUM_WORKERS,
    seed=SEED,
    project="runs/lab3",
    name="airplane_yolov8s",
    exist_ok=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    plots=True
)

run_dir = Path(train_results.save_dir)
print("run_dir:", run_dir)

In [ ]:
history_path = run_dir / "results.csv"
history_df = pd.read_csv(history_path)
history_df.columns = history_df.columns.str.strip()

print(history_df.tail())
print(history_df.columns.tolist())

In [ ]:
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(history_df["epoch"], history_df["train/box_loss"], label="train_box_loss")
plt.plot(history_df["epoch"], history_df["val/box_loss"], label="val_box_loss")
plt.legend()
plt.title("Box loss")

plt.subplot(1, 3, 2)
plt.plot(history_df["epoch"], history_df["metrics/mAP50(B)"], label="mAP50")
plt.axhline(TARGET_MAP50, linestyle="--", label="target")
plt.legend()
plt.title("mAP50")

plt.subplot(1, 3, 3)
plt.plot(history_df["epoch"], history_df["metrics/precision(B)"], label="precision")
plt.plot(history_df["epoch"], history_df["metrics/recall(B)"], label="recall")
plt.legend()
plt.title("Precision / Recall")

plt.tight_layout()
plt.show()

In [ ]:
best_weights = run_dir / "weights" / "best.pt"

best_model = YOLO(str(best_weights))

In [ ]:
def summarize_detection_metrics(metrics):
    precision = float(metrics.box.mp)
    recall = float(metrics.box.mr)
    map50 = float(metrics.box.map50)
    map50_95 = float(metrics.box.map)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": map50,
        "mAP50-95": map50_95,
    }


def print_detection_metrics(split_name, metric_values):
    print("{} precision = {:.4f}".format(split_name, metric_values["precision"]))
    print("{} recall    = {:.4f}".format(split_name, metric_values["recall"]))
    print("{} f1        = {:.4f}".format(split_name, metric_values["f1"]))
    print("{} mAP50     = {:.4f}".format(split_name, metric_values["mAP50"]))
    print("{} mAP50-95  = {:.4f}".format(split_name, metric_values["mAP50-95"]))

    if metric_values["mAP50"] > TARGET_MAP50:
        print(f"{split_name} target metric passed")
    else:
        print(f"{split_name} target metric failed")


def evaluate_split(split_name):
    best_model.model.eval()
    metrics = best_model.val(
        data=str(DATA_YAML),
        split=split_name,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        workers=NUM_WORKERS,
        project="runs/lab3",
        name=f"airplane_yolov8s_{split_name}",
        plots=True
    )
    summary = summarize_detection_metrics(metrics)
    print_detection_metrics(split_name, summary)
    return metrics, summary

In [ ]:
val_metrics, val_summary = evaluate_split("val")
test_metrics, test_summary = evaluate_split("test")

report = {
    "target_metric": "mAP50",
    "target_value": TARGET_MAP50,
    "val_passed": val_summary["mAP50"] > TARGET_MAP50,
    "test_passed": test_summary["mAP50"] > TARGET_MAP50,
    "val": val_summary,
    "test": test_summary,
    "best_weights": str(best_weights),
}

report_path = run_dir / "lab3_metrics.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print(report_path)
print(json.dumps(report, indent=2))

In [ ]:
test_sample_images = test_images[:5]

pred_results = best_model.predict(
    source=[str(p) for p in test_sample_images],
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    project="runs/lab3",
    name="predictions",
    exist_ok=True,
    save=True
)

pred_dir = Path(pred_results[0].save_dir)
print(pred_dir)
print(list(pred_dir.glob("*"))[:10])

In [ ]:
for img_path in sorted(pred_dir.glob("*.jpg"))[:5]:
    display(Image(filename=str(img_path)))
